# Consolidate every TRN_ file into TRN_ONE.txt

Concatenates every `examples/TRN_*.txt` file into a single `TRN_ONE.txt` at the repository root. Run the cells in order from a fresh kernel.

**Output conventions**

- Files are concatenated in sorted-filename order, so the result is deterministic and reproducible. Note that this Python sort is ASCII/case-sensitive, so `TRN_The*` names precede `TRN_superintelligence*` and `TRN_the_case*`.
- Nothing is inserted between files by default: no header, no separator line, no blank line. Switch `INCLUDE_SEPARATOR` in section 6 to `True` if you want file boundaries marked.
- Content is preserved rather than tidied. Line endings are normalised to **CRLF**, a missing final newline is **added** so files cannot fuse together, and a trailing blank line is **kept** rather than stripped. Section 5 lists any file that needed either adjustment.
- The output is UTF-8 **without BOM**, and the consolidated file ends with CRLF.
- No Unicode normalisation is applied anywhere, because this repo's IPA accents are combining marks (U+0301/U+0300) that must never be NFC-folded into precomposed characters such as U+01FD.

**Sections**

1. Import required libraries
2. Set input directory and output file path
3. Find all `TRN_` files
4. Sort and prepare the file list
5. Read and concatenate file contents
6. Add file separators (optional, off by default)
7. Write `TRN_ONE.txt`
8. Verify consolidated output

In [12]:
# 1. Import Required Libraries
from pathlib import Path

In [13]:
# 2. Set Input Directory and Output File Path
def find_repo_root(start=None, max_parents=4):
    """Walk up from `start` until a directory containing examples/ is found."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents[:max_parents]]:
        if (candidate / "examples").is_dir():
            return candidate
    raise FileNotFoundError(f"no directory containing 'examples' found from {here}")

REPO_ROOT = find_repo_root()
EXAMPLES = REPO_ROOT / "examples"
OUTPUT = REPO_ROOT / "TRN_ONE.txt"

print(f"repo root : {REPO_ROOT}")
print(f"input dir : {EXAMPLES}")
print(f"output    : {OUTPUT}")

repo root : D:\REPOS\Transcribe\ipa-transcribe
input dir : D:\REPOS\Transcribe\ipa-transcribe\examples
output    : D:\REPOS\Transcribe\ipa-transcribe\TRN_ONE.txt


In [50]:
# 3. Find All TRN_ Files
trn_files = sorted(p for p in EXAMPLES.glob("TRN_*.txt") if p.is_file())

if not trn_files:
    raise FileNotFoundError(f"no TRN_*.txt files found in {EXAMPLES}")

print(f"found {len(trn_files)} TRN_ files:")
for p in trn_files:
    print(f"  {p.name}")

found 19 TRN_ files:
  TRN_AgentHackingHuggingFace.txt
  TRN_ClimbingWithoutARope.txt
  TRN_CognitiveLoad.txt
  TRN_Gaslighitng.txt
  TRN_Habituation.txt
  TRN_Move37.txt
  TRN_NicaraguanSignLanguage.txt
  TRN_superintelligence_risk.txt
  TRN_the_case_for_superintelligence.txt
  TRN_TheAlignmentProblem.txt
  TRN_TheAriane5RocketFailure.txt
  TRN_TheDiscoveryOfChess.txt
  TRN_TheEscalationOfCommitment.txt
  TRN_TheFlashCrash.txt
  TRN_TheMorrisWorm.txt
  TRN_TheSelfishGene.txt
  TRN_TheTherac25RadiationAccidents.txt
  TRN_TheTuringMachine.txt
  TRN_TheY2KNonEvent.txt


In [51]:
# 4. Sort and Prepare the File List
# Alphabetical by filename keeps the consolidation order deterministic.
trn_files = sorted(trn_files, key=lambda p: p.name)

names = [p.name for p in trn_files]
assert len(names) == len(set(names)), "duplicate filenames would make the order ambiguous"

print(f"{len(names)} files in consolidation order:")
for index, name in enumerate(names, 1):
    print(f"  {index:2}. {name}")

19 files in consolidation order:
   1. TRN_AgentHackingHuggingFace.txt
   2. TRN_ClimbingWithoutARope.txt
   3. TRN_CognitiveLoad.txt
   4. TRN_Gaslighitng.txt
   5. TRN_Habituation.txt
   6. TRN_Move37.txt
   7. TRN_NicaraguanSignLanguage.txt
   8. TRN_TheAlignmentProblem.txt
   9. TRN_TheAriane5RocketFailure.txt
  10. TRN_TheDiscoveryOfChess.txt
  11. TRN_TheEscalationOfCommitment.txt
  12. TRN_TheFlashCrash.txt
  13. TRN_TheMorrisWorm.txt
  14. TRN_TheSelfishGene.txt
  15. TRN_TheTherac25RadiationAccidents.txt
  16. TRN_TheTuringMachine.txt
  17. TRN_TheY2KNonEvent.txt
  18. TRN_superintelligence_risk.txt
  19. TRN_the_case_for_superintelligence.txt


In [52]:
# 5. Read and Concatenate File Contents
def to_crlf(text):
    """Normalise CRLF/LF/CR to CRLF and guarantee a final newline.

    Trailing blank lines are preserved: this notebook concatenates files, it does not
    clean them up. A final newline is added only when a file lacks one, so that the
    next file's first line can never fuse onto it.
    """
    flat = text.replace("\r\n", "\n").replace("\r", "\n")
    if not flat.endswith("\n"):
        flat += "\n"
    return flat.replace("\n", "\r\n")

parts = []
per_file = []
notes = []

for p in trn_files:
    # Read bytes and decode: text mode would silently translate CRLF to LF
    # (universal newlines) and would leave no trace of the original endings.
    raw = p.read_bytes().decode("utf-8")
    body = to_crlf(raw)

    if not raw.endswith(("\n", "\r")):
        notes.append(f"{p.name}: had no final newline - one was added")
    if raw.endswith(("\n\n", "\r\r", "\r\n\r\n")):
        notes.append(f"{p.name}: ends with a blank line - preserved as-is")

    parts.append(body)
    per_file.append((p.name, body.count("\r\n"), len(body.split())))

print(f"{'file':<42}{'lines':>7}{'tokens':>8}")
for name, lines, tokens in per_file:
    print(f"{name:<42}{lines:>7}{tokens:>8}")
print(f"{'TOTAL':<42}{sum(l for _, l, _ in per_file):>7}{sum(t for _, _, t in per_file):>8}")

if notes:
    print("\nfile-boundary notes (content preserved, not corrected):")
    for note in notes:
        print(f"  {note}")

file                                        lines  tokens
TRN_AgentHackingHuggingFace.txt                 7     505
TRN_ClimbingWithoutARope.txt                    6     462
TRN_CognitiveLoad.txt                          92    6568
TRN_Gaslighitng.txt                             8     798
TRN_Habituation.txt                             9     482
TRN_Move37.txt                                 13     929
TRN_NicaraguanSignLanguage.txt                  7     598
TRN_TheAlignmentProblem.txt                    49    2711
TRN_TheAriane5RocketFailure.txt                 6     472
TRN_TheDiscoveryOfChess.txt                    41    1626
TRN_TheEscalationOfCommitment.txt               9     928
TRN_TheFlashCrash.txt                           6     483
TRN_TheMorrisWorm.txt                           6     514
TRN_TheSelfishGene.txt                         16    1264
TRN_TheTherac25RadiationAccidents.txt           6     478
TRN_TheTuringMachine.txt                       14    1063
TRN_TheY2KNonE

In [53]:
# 6. Add File Separators
# Off by default: TRN_ONE.txt is pure data, matching the convention that TRN_ files
# hold one paragraph per line with nothing inserted. Set True to mark boundaries.
INCLUDE_SEPARATOR = False

if INCLUDE_SEPARATOR:
    parts = [f"===== {name} =====\r\n{body}" for (name, _, _), body in zip(per_file, parts)]
    print("separator lines inserted")
else:
    print("no separators inserted (pure concatenation)")

no separators inserted (pure concatenation)


In [54]:
# 7. Write TRN_ONE.txt
combined = "".join(parts)
OUTPUT.write_text(combined, encoding="utf-8", newline="")

print(f"wrote {OUTPUT}")
print(f"files: {len(per_file)}  lines: {combined.count(chr(13) + chr(10))}  bytes: {OUTPUT.stat().st_size}")

wrote D:\REPOS\Transcribe\ipa-transcribe\TRN_ONE.txt
files: 19  lines: 343  bytes: 225491


In [55]:
# 8. Verify Consolidated Output
raw = OUTPUT.read_bytes()
text = raw.decode("utf-8")
expected = "".join(parts)
source_lines = sum(l for _, l, _ in per_file)

checks = {
    "TRN_ONE.txt exists": OUTPUT.is_file(),
    "no UTF-8 BOM": not raw.startswith(b"\xef\xbb\xbf"),
    "no lone LF": raw.count(b"\n") == raw.count(b"\r\n"),
    "ends with CRLF": raw.endswith(b"\r\n"),
    "no precomposed ae-acute (U+01FD)": chr(0x1FD) not in text,
    "matches expected concatenation": text == expected,
    "line count equals sum of source lines": text.count("\r\n") == source_lines,
    "all source files processed": len(per_file) == len(trn_files),
}
for label, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")

assert all(checks.values()), "verification failed"

lines = text.split("\r\n")
print(f"\nOK - {len(trn_files)} files, {len(lines) - 1} lines, {len(text.split())} tokens, {len(raw)} bytes")
print(f"first line: {lines[0][:70]}")
print(f"last line : {lines[-2][:70]}")

[PASS] TRN_ONE.txt exists
[PASS] no UTF-8 BOM
[PASS] no lone LF
[PASS] ends with CRLF
[PASS] no precomposed ae-acute (U+01FD)
[PASS] matches expected concatenation
[PASS] line count equals sum of source lines
[PASS] all source files processed

OK - 19 files, 343 lines, 23988 tokens, 225491 bytes
first line: ðij éjdʒənts ðət ɪskéjpt ðə bɒ́ks
last line : hɪ́strij ɪz fʊ́l əv píjpəl húw wər sɜ́rtən ðət ðə nɛ́kst tɛ̀knɒ́lədʒij
